In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\KPMG sem3\Project\Online Retail.xlsx")

In [3]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [4]:
print('shape of dataset')
print(df.shape)

print('columns of dataset')
print(df.columns)


shape of dataset
(541909, 8)
columns of dataset
Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [6]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
df['Description']=df['Description'].fillna(df['Description'].mode()[0])
df['CustomerID']=df['CustomerID'].fillna(df['CustomerID'].mode()[0])

In [8]:
df.duplicated().sum()

np.int64(5268)

In [9]:
df= df.drop_duplicates(inplace=True)

In [ ]:

df.shape
df.duplicated().sum()

AttributeError: 'NoneType' object has no attribute 'head'

In [12]:
# Convert date column
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDate"].min(), df["InvoiceDate"].max()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Clean invalid rows
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
df = df.dropna(subset=["CustomerID"]).copy()

# Create total sales column
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]
df.head()

In [ ]:
# Summary statistics
df[["Quantity", "UnitPrice", "TotalAmount"]].describe()

In [ ]:
# Check customer-level distribution
customer_summary = (
    df.groupby("CustomerID")
      .agg(
          TotalOrders=("InvoiceNo", "nunique"),
          TotalSpend=("TotalAmount", "sum"),
          LastPurchase=("InvoiceDate", "max"),
          AvgOrderValue=("TotalAmount", "mean")
      )
      .reset_index()
)

customer_summary.head()

In [ ]:
# Visualize monthly sales trend
monthly_sales = df.groupby(df["InvoiceDate"].dt.to_period("M"))["TotalAmount"].sum()
monthly_sales.plot(kind="bar", figsize=(14, 5), color="steelblue")
plt.title("Monthly Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Distribution of spending
sns.histplot(customer_summary["TotalSpend"], bins=30, kde=True)
plt.title("Customer Spending Distribution")
plt.show()

In [ ]:
# Boxplot to spot extreme values
sns.boxplot(x=customer_summary["TotalSpend"])
plt.title("Outlier Check for Total Spend")
plt.show()

In [ ]:
# Top customers
customer_summary.sort_values("TotalSpend", ascending=False).head(10)